# Fase 1 · Auditoría y visualización de BreastDCEDL
Cuaderno de trabajo de Álvaro Santamaría Antón. Ejecutar con el entorno **cancer**, desde la raíz o desde `notebooks/`.

Exploramos train; test solo participa en comprobaciones estructurales. No entrenamos, elegimos transformaciones ni eliminamos filas. Las funciones están en `scripts/auditoria_datos.py` para reutilizarlas en otros notebooks. Los resultados se guardan en `reports/local/auditoria/`, excluido de Git.

Fuentes: `GUIA.md`, `documentation/caso_breastdcedl.pdf` y la aclaración original de cortes aportada por Álvaro. Las instrucciones de esos documentos describen el trabajo completo; aquí ejecutamos únicamente la fase 1 autorizada.

In [ ]:
from pathlib import Path
import sys, json
import matplotlib.pyplot as plt
actual = Path.cwd().resolve()
RAIZ = next((p for p in (actual, *actual.parents) if (p / 'metadata/samples.csv').is_file()), actual)
assert (RAIZ / 'metadata/samples.csv').is_file(), 'Abre desde la raíz o notebooks/'
sys.path.insert(0, str(RAIZ))
from scripts import auditoria_datos as ad
SALIDA = RAIZ / 'reports/local/auditoria'
SALIDA.mkdir(parents=True, exist_ok=True)
pacientes, muestras = ad.cargar(RAIZ)
print('Python:', sys.executable)
print('Huellas de los CSV:', ad.huellas(RAIZ))

## 1. Integridad y separación por paciente
Cada fila de patients representa una paciente; cada fila de samples, un corte con tres fases. La etiqueta pertenece a la paciente. Comprobamos que sus cortes comparten split, fold y etiqueta y que ambas tablas coinciden.

Se reutiliza la decodificación previa de 38.109 PNG, registrada en `reports/local/dataset.json`: no repetimos toda esa lectura. La comprobación actual verifica rutas, nombres y presencia. No se ha buscado duplicación por contenido ni identidad clínica entre identificadores distintos.

In [ ]:
controles = ad.auditar(pacientes, muestras, RAIZ)
print(controles.to_string())
controles.to_csv(SALIDA / 'controles.csv')
assert controles.all(), 'Revisar controles fallidos antes de continuar'
print(muestras.groupby('split').agg(pacientes=('patient_id','nunique'), cortes=('sample_id','size')).to_string())

## 2. Clases, cohortes, folds y datos ausentes: solo train
Comparamos recuentos por paciente y por corte. Diez cortes de una misma paciente están correlacionados: no equivalen a diez personas independientes. El conjunto público tiene 1.273 pacientes, distinto de las 2.070 originales del artículo y de las 177 reservadas al profesor.

Los nulos clínicos significan datos no disponibles; no se rellenan con cero. Las tablas no seleccionan variables de entrada.

In [ ]:
tablas = ad.tablas_train(pacientes, muestras)
for nombre, tabla in tablas.items():
    print(); print(nombre); print(tabla.to_string())
    tabla.to_csv(SALIDA / (nombre + '.csv'))
fig = ad.figura_clases(tablas)
fig.savefig(SALIDA / 'clases_train.png', dpi=140)
plt.show()

### Lectura de los resultados
- Train: 775 pacientes pCR=0 y 322 pCR=1 (29,35 % positivas). Por corte: 7.729 y 3.216. Predecir siempre 0 daría aproximadamente 70,65 % de accuracy por paciente de train; no demuestra aprendizaje.
- Cohortes: Duke 44/209 positivas (21,05 %), I-SPY1 26/104 (25 %) e I-SPY2 252/784 (32,14 %). Estas diferencias descriptivas pueden permitir atajos ligados a la cohorte; no prueban causalidad ni que una red ya haya aprendido ese sesgo.
- Los folds contienen 2.186, 2.190, 2.192, 2.192 y 2.185 cortes. El CSV resuelve la discrepancia de la guía: fold 0 tiene **2.186**, no 2.187. No se modifican folds.
- 1.091 pacientes de train aportan diez cortes; dos aportan cinco, tres seis y una siete. Ninguna se excluye por ello.
- La comparación con límites originales es descriptiva: Duke coincide en 209/209; I-SPY1 en 31/104 e I-SPY2 en 150/784. Los índices de I-SPY pertenecen al volumen recortado. **No filtrar por mask_start/end.**

## 3. PRE, EARLY, LATE y EARLY − PRE
Seleccionamos el primer identificador ordenado de cada combinación cohorte/clase y el corte mediano de los disponibles. Son seis ejemplos para inspeccionar, no una muestra representativa.

Cada fase usa la misma escala [0,1], obtenida por división entre 255. La resta se hace en float32 para conservar valores negativos. El realce usa una escala divergente fija [-1,1], sin ocultar los negativos ni reescalar cada imagen. Esto es visualización, no una decisión de normalización para el modelo.

Observar si las estructuras ocupan posiciones comparables entre fases. La correspondencia de nombres y tamaño no certifica registro anatómico perfecto. La media de toda la imagen incluye fondo y no equivale a una medida del tumor.

In [ ]:
ejemplos = ad.ejemplos_train(pacientes, muestras)
print(ejemplos[['sample_id','dataset','pCR']].to_string(index=False))
fig, rangos = ad.figura_fases(ejemplos, RAIZ)
fig.savefig(SALIDA / 'fases_train.png', dpi=140)
rangos.to_csv(SALIDA / 'rangos_ejemplos.csv', index=False)
print(rangos.to_string(index=False))
plt.show()

## 4. Una paciente, varias alturas
Estos cortes pertenecen a una sola paciente y a una misma fase EARLY. z ordena posiciones; no representa tiempo. Puede haber saltos porque no se entregan todos los cortes del volumen.

Puedes cambiar `pid` por otro identificador de train. El explorador rechaza test.

In [ ]:
pid = 'ISPY1_1001'
fig = ad.figura_paciente(muestras, pid, RAIZ)
fig.savefig(SALIDA / 'cortes_paciente_train.png', dpi=140)
plt.show()

## 5. Interpretación y límites
Una entrada es un tensor de forma (3,256,256), con PRE/EARLY/LATE en ese orden. pCR=1 indica respuesta patológica completa tras el tratamiento; la imagen se tomó antes del tratamiento. No son canales RGB.

La selección de cortes difiere entre Duke e I-SPY y las proporciones de clase también difieren entre cohortes. Son posibles fuentes de sesgo que habrá que considerar en la evaluación posterior. Aquí no elegimos arquitectura, pérdidas concretas por fold, agregación ni umbral.

La guía dice que PRE < EARLY se cumple en todas las pacientes, pero el enunciado refiere una comprobación en **120 pacientes**. No extrapolamos esa cifra a toda la población ni exigimos una desigualdad píxel a píxel. Los seis ejemplos tampoco verifican una frecuencia general de washout.

Pendiente de comentar con Álvaro: qué observamos entre fases, por qué separar por paciente y cómo afectan cohortes/desbalance. La fase 2 requiere acordar su protocolo; no comienza al ejecutar este cuaderno.